# Autoencoder + KMeans 고객 군집 분석

소비 패턴을 Autoencoder로 압축 후 KMeans로 고객 유형을 분류합니다.

In [ ]:
import os, glob, gc, time
import numpy as np
import pandas as pd
import joblib
import torch
import torch.nn as nn
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

DATA_DIR   = "/kaggle/input/gyeonggi-card-data"
OUTPUT_DIR = "/kaggle/working"
os.makedirs(f"{OUTPUT_DIR}/model", exist_ok=True)

N_CLUSTERS  = 5
SAMPLE_SIZE = 500000
EPOCHS      = 30
LATENT_DIM  = 8

files = sorted(glob.glob(f"{DATA_DIR}/tbsh_gyeonggi_day_*.csv"))
print(f"파일 수: {len(files)}개")

In [ ]:
# 1) 데이터 로드 및 샘플링
USE_COLS = ["age", "hour", "day", "sex", "card_tpbuz_nm_2", "amt", "cnt"]
dfs = []
per_file = max(1000, SAMPLE_SIZE // len(files))
for f in files:
    df = pd.read_csv(f, encoding="utf-8-sig", usecols=USE_COLS)
    df = df.dropna()
    if len(df) > per_file:
        df = df.sample(per_file, random_state=42)
    dfs.append(df)

df_all = pd.concat(dfs, ignore_index=True)
del dfs; gc.collect()
print(f"총 {len(df_all):,}행")

In [ ]:
# 2) 인코딩
le_sex  = LabelEncoder()
le_biz2 = LabelEncoder()
df_all["sex"]             = le_sex.fit_transform(df_all["sex"].astype(str))
df_all["card_tpbuz_nm_2"] = le_biz2.fit_transform(df_all["card_tpbuz_nm_2"].astype(str))

FEATURES = ["age", "hour", "day", "sex", "card_tpbuz_nm_2", "cnt"]
scaler = MinMaxScaler()
X = scaler.fit_transform(df_all[FEATURES].astype(float))
X_tensor = torch.tensor(X, dtype=torch.float32)
print(f"피처 수: {X.shape[1]}")

In [ ]:
# 3) Autoencoder 정의
class Autoencoder(nn.Module):
    def __init__(self, input_dim, latent_dim):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 32), nn.ReLU(),
            nn.Linear(32, 16),        nn.ReLU(),
            nn.Linear(16, latent_dim)
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 16), nn.ReLU(),
            nn.Linear(16, 32),         nn.ReLU(),
            nn.Linear(32, input_dim),  nn.Sigmoid()
        )
    def forward(self, x):
        return self.decoder(self.encoder(x))
    def encode(self, x):
        return self.encoder(x)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ae     = Autoencoder(X.shape[1], LATENT_DIM).to(device)
opt    = torch.optim.Adam(ae.parameters(), lr=0.001)
crit   = nn.MSELoss()
print(f"Device: {device}")

In [ ]:
# 4) Autoencoder 학습
from torch.utils.data import DataLoader, TensorDataset

loader = DataLoader(TensorDataset(X_tensor.to(device)), batch_size=512, shuffle=True)
t0 = time.time()
for epoch in range(1, EPOCHS+1):
    ae.train()
    losses = []
    for (xb,) in loader:
        opt.zero_grad()
        loss = crit(ae(xb), xb)
        loss.backward()
        opt.step()
        losses.append(loss.item())
    if epoch % 10 == 0:
        print(f"Epoch {epoch:3d}  loss={np.mean(losses):.6f}  ({time.time()-t0:.0f}s)")

In [ ]:
# 5) 잠재 공간 추출 → KMeans 군집
ae.eval()
with torch.no_grad():
    latent = ae.encode(X_tensor.to(device)).cpu().numpy()

kmeans = KMeans(n_clusters=N_CLUSTERS, random_state=42, n_init=10)
labels = kmeans.fit_predict(latent)
df_all["cluster"] = labels

# PCA 2D for visualization
pca = PCA(n_components=2, random_state=42)
latent_2d = pca.fit_transform(latent[:50000])  # 시각화용 샘플
centers_2d = pca.transform(kmeans.cluster_centers_)

print("군집별 크기:", pd.Series(labels).value_counts().sort_index().to_dict())

In [ ]:
# 6) 군집별 특성 요약
AGE_MAP  = {1:"10대",2:"20대",3:"20대",4:"30대",5:"40대",6:"50대",7:"60대",8:"70대",9:"80대",10:"90대",11:"100세+"}
DAY_MAP  = {1:"월",2:"화",3:"수",4:"목",5:"금",6:"토",7:"일"}
HOUR_MAP = {1:"새벽",2:"아침",3:"오전",4:"점심",5:"오후",6:"늦오후",7:"저녁",8:"밤",9:"늦은밤",10:"심야"}

stats_rows = []
ratios = []
names  = []
for c in range(N_CLUSTERS):
    sub = df_all[df_all["cluster"] == c]
    top_age  = AGE_MAP.get(int(sub["age"].mode()[0]),  str(sub["age"].mode()[0]))
    top_day  = DAY_MAP.get(int(sub["day"].mode()[0]),  str(sub["day"].mode()[0]))
    top_hour = HOUR_MAP.get(int(sub["hour"].mode()[0]),str(sub["hour"].mode()[0]))
    top_sex  = "여성" if sub["sex"].mode()[0] == 1 else "남성"
    top_biz  = le_biz2.inverse_transform([int(sub["card_tpbuz_nm_2"].mode()[0])])[0]
    ratio    = round(len(sub) / len(df_all) * 100, 1)
    name     = f"{top_age} {top_sex} {top_hour}형"
    names.append(name)
    ratios.append(ratio)
    stats_rows.append({
        "군집": name, "비율(%)": ratio,
        "주요 연령": top_age, "성별": top_sex,
        "주요 요일": top_day, "주요 시간대": top_hour,
        "선호 업종": top_biz,
        "평균 매출(원)": int(sub["amt"].mean()) if "amt" in sub.columns else "-",
        "평균 거래건수": round(sub["cnt"].mean(), 1),
    })
    print(f"군집 {c}: {name} ({ratio}%)")

stats_df = pd.DataFrame(stats_rows)

In [ ]:
# 7) 저장
ae.cpu()
joblib.dump({
    "model":   ae,
    "scaler":  scaler,
    "kmeans":  kmeans,
    "pca":     pca,
    "labels":  labels,
    "centers": centers_2d,
    "stats":   stats_df,
    "names":   names,
    "ratios":  ratios,
    "n_clusters": N_CLUSTERS,
}, f"{OUTPUT_DIR}/model/cluster_model.pkl")
print("저장 완료: model/cluster_model.pkl")